# 🧬 Biopython Masterclass: Basic to Advanced
## A Practical Bioinformatics Notebook

---

### What is Biopython?
Biopython is the Swiss-army knife of computational biology in Python. It provides tools for:
- Parsing biological file formats (FASTA, FASTQ, GenBank, PDB...)
- Sequence analysis and manipulation
- Accessing online biological databases (NCBI, UniProt, PDB)
- Running bioinformatics tools (BLAST, Clustal...)
- Phylogenetics, structural biology, population genetics

### Notebook Structure
| Section | Topics |
|---|---|
| **Part 1** | Installation & Setup |
| **Part 2** | Sequences — The Foundation |
| **Part 3** | Sequence I/O — Reading & Writing Files |
| **Part 4** | Sequence Analysis |
| **Part 5** | Accessing NCBI Databases (Entrez) |
| **Part 6** | BLAST — Sequence Similarity Search |
| **Part 7** | Multiple Sequence Alignment |
| **Part 8** | Phylogenetics |
| **Part 9** | Structural Bioinformatics (PDB) |
| **Part 10** | Population Genetics |
| **Part 11** | Real-World Mini-Pipeline |

> **Biological context woven throughout** — every tool is tied to a real-world use case.

---
## Part 1: Installation & Setup

In [ ]:
# Install Biopython (run once)
# !pip install biopython

# Verify installation
import Bio
print(f"Biopython version: {Bio.__version__}")

# Core imports we will use throughout
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO, Entrez, Align, AlignIO, Phylo
from Bio.Blast import NCBIWWW, NCBIXML
import warnings
warnings.filterwarnings('ignore')
print("All imports successful ✓")

---
## Part 2: Sequences — The Foundation

### 🧬 Biological Context
Every organism's genetic information is encoded in DNA (or RNA for some viruses) as a sequence of nucleotides (A, T, G, C). Biopython's `Seq` object is your starting point for working with this information.

### Analogy
Think of a `Seq` object like a Python string — but *biology-aware*. It knows the alphabet (DNA, RNA, protein) and has built-in methods for biological operations.

In [ ]:
# ─────────────────────────────────────────────
# 2.1  Creating Sequence Objects
# ─────────────────────────────────────────────

# DNA sequence (example: fragment of E. coli 16S rRNA gene)
dna_seq = Seq("ATGGCAAGCTTAAGCTTACGATCGATCGATCGATCGTAGCATCG")
print(f"DNA Sequence : {dna_seq}")
print(f"Type         : {type(dna_seq)}")
print(f"Length       : {len(dna_seq)} bp")

In [ ]:
# ─────────────────────────────────────────────
# 2.2  Seq behaves like a Python string
# ─────────────────────────────────────────────

# Indexing (0-based)
print(f"First base        : {dna_seq[0]}")
print(f"Last base         : {dna_seq[-1]}")

# Slicing
print(f"First 10 bases    : {dna_seq[:10]}")
print(f"Bases 5–15        : {dna_seq[5:15]}")

# Iteration
base_counts = {}
for base in dna_seq:
    base_counts[base] = base_counts.get(base, 0) + 1
print(f"Base counts       : {base_counts}")

# Concatenation
seq2 = Seq("TTTTAAAA")
combined = dna_seq + seq2
print(f"Concatenated len  : {len(combined)} bp")

In [ ]:
# ─────────────────────────────────────────────
# 2.3  Core Molecular Biology Operations
# ─────────────────────────────────────────────

coding_seq = Seq("ATGAAACCCGGGTTTTAA")

# Complement — pairs each base with its partner (A↔T, G↔C)
comp = coding_seq.complement()
print(f"Original    : {coding_seq}")
print(f"Complement  : {comp}")

# Reverse complement — needed because DNA is anti-parallel
# This is what you use to get the sequence on the OTHER strand
rev_comp = coding_seq.reverse_complement()
print(f"Rev. comp.  : {rev_comp}")

# Transcription — DNA → RNA (T is replaced by U)
mrna = coding_seq.transcribe()
print(f"mRNA        : {mrna}")

# Translation — RNA → Protein (uses the standard genetic code)
protein = coding_seq.translate()
print(f"Protein     : {protein}")

# Stop codons are represented as '*'
# The last '*' means this is a complete ORF

In [ ]:
# ─────────────────────────────────────────────
# 2.4  Translation Options & Genetic Codes
# ─────────────────────────────────────────────

# Stop at first stop codon — important for ORF extraction
protein_clean = coding_seq.translate(to_stop=True)
print(f"Protein (no stop) : {protein_clean}")

# Mitochondrial genetic code (table 2) — used for mt genomes!
# TGA encodes Trp in mitochondria, not Stop
mt_seq = Seq("ATGTGACCC")  # TGA = Stop in standard, Trp in mt
std_translation = mt_seq.translate(table=1)
mt_translation  = mt_seq.translate(table=2)
print(f"Standard code   : {std_translation}")
print(f"Mitochondrial   : {mt_translation}")

# Bacterial genetic code (table 11)
bacterial_protein = coding_seq.translate(table=11, to_stop=True)
print(f"Bacterial code  : {bacterial_protein}")

In [ ]:
# ─────────────────────────────────────────────
# 2.5  GC Content & Sequence Statistics
# ─────────────────────────────────────────────
# GC content is biologically important:
# - Higher GC → more stable DNA (3 H-bonds vs 2 for AT)
# - Used for primer design, species identification, AMR gene analysis

from Bio.SeqUtils import gc_fraction, MeltingTemp

primer = Seq("GCGCATCGATCGATCGCGCG")
gc = gc_fraction(primer)
print(f"Primer        : {primer}")
print(f"GC content    : {gc:.1%}")

# Melting temperature — critical for PCR primer design
tm_basic  = MeltingTemp.Tm_Wallace(primer)   # Wallace rule (simple)
tm_nn     = MeltingTemp.Tm_NN(primer)        # Nearest-neighbor (more accurate)
print(f"Tm (Wallace)  : {tm_basic:.1f} °C")
print(f"Tm (NN)       : {tm_nn:.1f} °C")

# Molecular weight of a sequence
from Bio.SeqUtils import molecular_weight
mw = molecular_weight(primer, "DNA")
print(f"MW (DNA)      : {mw:.1f} Da")

In [ ]:
# ─────────────────────────────────────────────
# 2.6  SeqRecord — Sequences with Metadata
# ─────────────────────────────────────────────
# Real biological sequences always come with metadata.
# SeqRecord bundles a Seq with an ID, name, description, and features.

record = SeqRecord(
    Seq("ATGAAACCCGGGTTTTAA"),
    id="EC_K12_0001",
    name="thrA_fragment",
    description="E. coli K-12 thrA gene fragment | aspartate kinase",
    annotations={"organism": "Escherichia coli K-12",
                 "mol_type": "genomic DNA",
                 "source": "NCBI"}
)

print(record)
print(f"\nID          : {record.id}")
print(f"Name        : {record.name}")
print(f"Description : {record.description}")
print(f"Sequence    : {record.seq}")
print(f"Annotations : {record.annotations}")

---
## Part 3: Sequence I/O — Reading & Writing Files

### 🧬 Biological Context
Bioinformatics data comes in many file formats. The most common are:
- **FASTA** — sequences + descriptions (reference genomes, protein databases)
- **FASTQ** — sequences + quality scores (raw sequencing reads from Illumina/Nanopore)
- **GenBank** — sequences + rich annotations (genes, features, publications)
- **EMBL** — similar to GenBank, used by European databases

In [ ]:
import os
from io import StringIO

# ─────────────────────────────────────────────
# 3.1  Writing FASTA files
# ─────────────────────────────────────────────

# Create sample records representing pathogen genes
records = [
    SeqRecord(Seq("ATGAAAGCAATTTTCGTACTGAAAGGTTTTGTTGGTTTTCTTAAATTTGAA"),
              id="AMP_R_001", description="Beta-lactamase TEM-1 | ampicillin resistance"),
    SeqRecord(Seq("ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTTTTGCGGCATT"),
              id="AMP_R_002", description="Beta-lactamase SHV-1 | ampicillin resistance"),
    SeqRecord(Seq("ATGCAGCAGCAGGCGATCTTGAACCTGACAGACAGTCTGGCGCTGGTGAT"),
              id="VAN_R_001", description="VanA | vancomycin resistance"),
]

# Write to FASTA
fasta_path = "/tmp/amr_genes.fasta"
with open(fasta_path, "w") as f:
    SeqIO.write(records, f, "fasta")
print(f"Written {len(records)} records to {fasta_path}")

# Preview the file
with open(fasta_path) as f:
    print(f.read())

In [ ]:
# ─────────────────────────────────────────────
# 3.2  Reading FASTA files
# ─────────────────────────────────────────────

# Method 1: Read all records into a list
records_in = list(SeqIO.parse(fasta_path, "fasta"))
print(f"Records parsed: {len(records_in)}")

for rec in records_in:
    gc = gc_fraction(rec.seq)
    print(f"  {rec.id:15} | {len(rec.seq):4} bp | GC: {gc:.1%} | {rec.description[:40]}")

print()
# Method 2: Iterator (memory-efficient for large files like reference genomes)
print("Iterating (memory-efficient):")
for rec in SeqIO.parse(fasta_path, "fasta"):
    print(f"  Processing: {rec.id}")

In [ ]:
# ─────────────────────────────────────────────
# 3.3  FASTQ — Raw Sequencing Reads
# ─────────────────────────────────────────────
# FASTQ is the output format of NGS sequencers (Illumina, Oxford Nanopore).
# Each read has: @ID, Sequence, +, Quality scores (Phred encoded)

# Create a mock FASTQ file
fastq_content = """\
@READ_001 length=50
GCATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@READ_002 length=50
ATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATGATG
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@READ_003 length=50
TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTT
+
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
"""

fastq_path = "/tmp/sample_reads.fastq"
with open(fastq_path, "w") as f:
    f.write(fastq_content)

# Parse and inspect
print(f"{'Read ID':<15} {'Length':>6} {'Avg Quality':>12} {'Status':<10}")
print("-" * 50)
for rec in SeqIO.parse(fastq_path, "fastq"):
    quals = rec.letter_annotations["phred_quality"]
    avg_q = sum(quals) / len(quals)
    # Phred quality: Q30 = 99.9% accuracy; Q20 = 99%; <Q20 = poor
    status = "PASS" if avg_q >= 20 else "FAIL"
    print(f"{rec.id:<15} {len(rec.seq):>6} {avg_q:>12.1f} {status:<10}")

In [ ]:
# ─────────────────────────────────────────────
# 3.4  Reading GenBank records
# ─────────────────────────────────────────────
# GenBank format is the richest — it contains sequence + annotations
# (genes, CDSs, regulatory elements, publications, taxonomy)

# We'll use Entrez to fetch a real record (needs internet)
# For offline demo, we use a mock GenBank string

gb_mock = """\
LOCUS       EC_LACZ                  100 bp    DNA     linear   BCT 01-JAN-2024
DEFINITION  Escherichia coli lacZ gene fragment (beta-galactosidase).
ACCESSION   EC_LACZ
VERSION     EC_LACZ.1
KEYWORDS    lacZ; beta-galactosidase; E.coli.
SOURCE      Escherichia coli K-12
  ORGANISM  Escherichia coli K-12
            Bacteria; Proteobacteria; Gammaproteobacteria; Enterobacterales;
            Enterobacteriaceae; Escherichia.
FEATURES             Location/Qualifiers
     gene            1..100
                     /gene="lacZ"
     CDS             1..100
                     /gene="lacZ"
                     /product="beta-galactosidase"
                     /protein_id="EC_LACZ_p1"
                     /translation="MAAA"
ORIGIN
        1 atgaaagcaa ttttcgtact gaaaggtttt gttggttttc ttaaatttga acgtcatgcg
       61 catcgatcga tcgatcgatc gatcgatcga tcgatcgatc
//
"""

gb_path = "/tmp/ec_lacz.gb"
with open(gb_path, "w") as f:
    f.write(gb_mock)

for rec in SeqIO.parse(gb_path, "genbank"):
    print(f"ID          : {rec.id}")
    print(f"Name        : {rec.name}")
    print(f"Description : {rec.description}")
    print(f"Length      : {len(rec.seq)} bp")
    print(f"Organism    : {rec.annotations.get('organism', 'N/A')}")
    print(f"\nFeatures ({len(rec.features)}):")
    for feat in rec.features:
        print(f"  [{feat.type}] Location: {feat.location} | Qualifiers: {dict(feat.qualifiers)}")

In [ ]:
# ─────────────────────────────────────────────
# 3.5  Format Conversion
# ─────────────────────────────────────────────
# Real task: convert GenBank → FASTA (e.g., to use as BLAST database)

converted_path = "/tmp/ec_lacz_converted.fasta"
count = SeqIO.convert(gb_path, "genbank", converted_path, "fasta")
print(f"Converted {count} record(s) to FASTA")

with open(converted_path) as f:
    print(f.read())

# Dictionary index — fast random access to large FASTA files
# Critical for working with reference genomes (e.g., hg38, GRCh38)
seq_dict = SeqIO.to_dict(SeqIO.parse(fasta_path, "fasta"))
print(f"\nDictionary keys: {list(seq_dict.keys())}")
print(f"Fetching AMP_R_001: {seq_dict['AMP_R_001'].seq[:20]}...")

---
## Part 4: Sequence Analysis

### 🧬 Biological Context
Sequence analysis is at the heart of bioinformatics — from identifying ORFs in a pathogen genome to designing PCR primers for AMR gene detection.

In [ ]:
# ─────────────────────────────────────────────
# 4.1  Finding Open Reading Frames (ORFs)
# ─────────────────────────────────────────────
# An ORF starts with ATG (Met) and ends with a stop codon (TAA/TAG/TGA)
# Finding ORFs is the first step in genome annotation

def find_orfs(sequence, min_length=30):
    """Find all ORFs in all 6 reading frames."""
    orfs = []
    seq = Seq(sequence) if isinstance(sequence, str) else sequence
    
    # 6 frames: 3 forward, 3 reverse complement
    frames = [
        (seq,              "+", 0),
        (seq[1:],          "+", 1),
        (seq[2:],          "+", 2),
        (seq.reverse_complement(), "-", 0),
        (seq.reverse_complement()[1:], "-", 1),
        (seq.reverse_complement()[2:], "-", 2),
    ]
    
    for frame_seq, strand, offset in frames:
        aa_seq = frame_seq.translate()
        aa_str = str(aa_seq)
        
        # Split by stop codon
        peptides = aa_str.split("*")
        pos = 0
        for pep in peptides:
            if "M" in pep:  # Must contain Met (start codon)
                start_aa = pep.index("M")
                orf_pep = pep[start_aa:]
                if len(orf_pep) * 3 >= min_length:
                    orfs.append({
                        "strand": strand, "frame": offset + 1,
                        "aa_length": len(orf_pep),
                        "nt_length": len(orf_pep) * 3,
                        "protein": orf_pep
                    })
            pos += len(pep) + 1  # +1 for stop codon
    return sorted(orfs, key=lambda x: x["nt_length"], reverse=True)

# Test on an AMR gene sequence
tem1_frag = ("ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTTTTGCGGCATTTTGCCTTCCT"
             "GTTTTTGCTCACCCAGAAACGCTGGTAAAAGTATTTAATCTTTTTAATCAACAGGATTTG")
orfs = find_orfs(tem1_frag, min_length=30)
print(f"Found {len(orfs)} ORF(s) ≥ 30 nt:")
for i, orf in enumerate(orfs[:5], 1):
    print(f"  ORF {i}: strand={orf['strand']} frame={orf['frame']} "
          f"len={orf['nt_length']}nt protein={orf['protein'][:15]}...")

In [ ]:
# ─────────────────────────────────────────────
# 4.2  Sequence Search & Pattern Matching
# ─────────────────────────────────────────────
# Find restriction enzyme sites — critical for cloning
# Find motifs — e.g., Shine-Dalgarno sequence, TATA box, AMR signatures

from Bio.Seq import Seq

genome_fragment = Seq(
    "GCATCGATCGAATTCATCGATCGATCGAAGCTTCGATCGATCGCTCGAGCATCG"
)

# Common restriction sites
restriction_sites = {
    "EcoRI": "GAATTC",
    "HindIII": "AAGCTT",
    "XhoI": "CTCGAG",
    "BamHI": "GGATCC",
}

print(f"Sequence: {genome_fragment}\n")
print(f"{'Enzyme':<12} {'Site':<10} {'Positions (0-based)'}")
print("-" * 45)
for enzyme, site in restriction_sites.items():
    positions = []
    start = 0
    while True:
        pos = str(genome_fragment).find(site, start)
        if pos == -1:
            break
        positions.append(pos)
        start = pos + 1
    count = len(positions)
    pos_str = str(positions) if positions else "not found"
    print(f"{enzyme:<12} {site:<10} {pos_str}")

In [ ]:
# ─────────────────────────────────────────────
# 4.3  Pairwise Sequence Alignment
# ─────────────────────────────────────────────
# Alignment tells us HOW SIMILAR two sequences are and WHERE they differ.
# Used for: SNP calling, gene family analysis, AMR gene classification

from Bio import Align

aligner = Align.PairwiseAligner()

# ── Global alignment (Needleman-Wunsch) ──
# Use when: comparing full-length genes or proteins
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -2
aligner.extend_gap_score = -0.5

# Comparing two beta-lactamase variants (TEM-1 vs TEM-2 fragments)
tem1 = Seq("ATGAGTATTCAACATTTCCGTGTCGCCCTT")
tem2 = Seq("ATGAGTATTCAACATTTCCGTGTCGCCATT")  # one mismatch

alignments = aligner.align(tem1, tem2)
best = next(iter(alignments))
print("=== Global Alignment (TEM-1 vs TEM-2) ===")
print(best)
print(f"Score     : {best.score:.1f}")
identity = sum(a == b for a, b in zip(str(tem1), str(tem2))) / len(tem1) * 100
print(f"Identity  : {identity:.1f}%")

# ── Local alignment (Smith-Waterman) ──
# Use when: finding a gene within a longer contig
aligner.mode = "local"
long_seq = Seq("NNNNNNATGAGTATTCAACATTTCCGTGTCGCCCTTNNNNNN")
query    = Seq("ATGAGTATTCAACATTTCCGTGTCGCCCTT")
local_alignments = aligner.align(long_seq, query)
local_best = next(iter(local_alignments))
print("\n=== Local Alignment (finding gene in contig) ===")
print(local_best)
print(f"Score: {local_best.score:.1f}")

In [ ]:
# ─────────────────────────────────────────────
# 4.4  Amino Acid & Protein Analysis
# ─────────────────────────────────────────────

from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Analyze a protein sequence
protein_seq = "MASFKGIFAGLLFSIIASVGADQPAMAEAGRGIKRGYEYKDQKRPTPVSPQNMPNVFDAAKEKYPDLSPTRIIENPQHKYKTK"

analysis = ProteinAnalysis(protein_seq)

print("=== Protein Properties ===")
print(f"Length               : {len(protein_seq)} aa")
print(f"Molecular weight     : {analysis.molecular_weight():.1f} Da")
print(f"Isoelectric point    : pH {analysis.isoelectric_point():.2f}")
print(f"Instability index    : {analysis.instability_index():.2f}")
print(f"  (>40 = unstable; this affects recombinant protein production)")
print(f"GRAVY score          : {analysis.gravy():.3f}")
print(f"  (negative = hydrophilic / soluble; positive = membrane-associated)")
print(f"Aromaticity          : {analysis.aromaticity():.3f}")

# Secondary structure fractions
helix, turn, sheet = analysis.secondary_structure_fraction()
print(f"\nPredicted 2° structure:")
print(f"  Alpha helix  : {helix:.1%}")
print(f"  Beta sheet   : {sheet:.1%}")
print(f"  Turns        : {turn:.1%}")

# Amino acid composition
aa_comp = analysis.get_amino_acids_percent()
print(f"\nAmino acid composition (top 5):")
for aa, pct in sorted(aa_comp.items(), key=lambda x: -x[1])[:5]:
    print(f"  {aa}: {pct:.1%}")

---
## Part 5: Accessing NCBI Databases (Entrez)

### 🧬 Biological Context
NCBI (National Center for Biotechnology Information) is the world's largest repository of biological sequences. Biopython's `Entrez` module gives you programmatic access to:
- **GenBank/RefSeq** — genome sequences
- **PubMed** — scientific literature
- **SRA** — raw sequencing data
- **Taxonomy** — organism classification

In [ ]:
# ─────────────────────────────────────────────
# 5.1  Entrez Setup
# ─────────────────────────────────────────────
# IMPORTANT: Always set your email — NCBI uses this to contact you if there
# are problems with your queries, and to avoid IP banning for heavy usage.

Entrez.email = "your.email@example.com"  # ← Replace with yours!
Entrez.tool  = "BioinformaticsCourse"     # Optional but good practice

print("Entrez configured.")
print(f"Email : {Entrez.email}")
print(f"Tool  : {Entrez.tool}")

In [ ]:
# ─────────────────────────────────────────────
# 5.2  Searching NCBI (esearch)
# ─────────────────────────────────────────────
# esearch finds record IDs that match your query

try:
    # Search for beta-lactamase genes in Klebsiella pneumoniae
    # This is a clinically important AMR pathogen
    handle = Entrez.esearch(
        db="nucleotide",
        term="KPC beta-lactamase[Gene] AND Klebsiella pneumoniae[Organism]",
        retmax=10
    )
    record = Entrez.read(handle)
    handle.close()
    
    print(f"Total results : {record['Count']}")
    print(f"IDs returned  : {record['IdList']}")
    ncbi_ids = record['IdList']
    
except Exception as e:
    print(f"[Offline mode] Entrez requires internet. Error: {e}")
    ncbi_ids = ["CP044810"]  # Fallback ID for demos

In [ ]:
# ─────────────────────────────────────────────
# 5.3  Fetching Records (efetch)
# ─────────────────────────────────────────────

try:
    # Fetch the first 3 results
    handle = Entrez.efetch(
        db="nucleotide",
        id=",".join(ncbi_ids[:3]),
        rettype="gb",        # GenBank format
        retmode="text"
    )
    fetched_records = list(SeqIO.parse(handle, "genbank"))
    handle.close()
    
    print(f"Fetched {len(fetched_records)} GenBank records:")
    for rec in fetched_records:
        organism = rec.annotations.get('organism', 'Unknown')
        print(f"  {rec.id:<20} {len(rec.seq):>8} bp | {organism}")
        
except Exception as e:
    print(f"[Offline mode] Entrez requires internet. Error: {e}")

In [ ]:
# ─────────────────────────────────────────────
# 5.4  Extracting Features from GenBank Records
# ─────────────────────────────────────────────
# GenBank records contain annotated features: genes, CDSs, regulatory regions

# We'll use our mock record for this
gb_rich_mock = """\
LOCUS       KPC_DEMO                 100 bp    DNA     linear   BCT 01-JAN-2024
DEFINITION  KPC-2 beta-lactamase gene, Klebsiella pneumoniae.
ACCESSION   KPC_DEMO
VERSION     KPC_DEMO.1
FEATURES             Location/Qualifiers
     gene            1..100
                     /gene="blaKPC-2"
                     /note="AMR gene; carbapenem resistance"
     CDS             1..100
                     /gene="blaKPC-2"
                     /product="KPC-2 beta-lactamase"
                     /antibiotic_resistance="carbapenem"
                     /protein_id="KPC_p1"
                     /translation="MSIQHFRVALIPFFAAFCLPVFA"
     misc_feature    10..30
                     /note="Signal peptide region"
ORIGIN
        1 atgagcattc agtttcgtgt tgcactgatc ccatttttcg cagcgttttg cttatcccca
       61 gtcttcgctg cacccagaaa cgctggtaaa agtatttaat
//
"""

gb_rich_path = "/tmp/kpc_demo.gb"
with open(gb_rich_path, "w") as f:
    f.write(gb_rich_mock)

for rec in SeqIO.parse(gb_rich_path, "genbank"):
    print(f"Record: {rec.id} — {rec.description}")
    print(f"Total features: {len(rec.features)}\n")
    
    for feat in rec.features:
        if feat.type == "CDS":
            gene    = feat.qualifiers.get("gene", ["unknown"])[0]
            product = feat.qualifiers.get("product", ["unknown"])[0]
            prot_id = feat.qualifiers.get("protein_id", ["N/A"])[0]
            print(f"CDS Found:")
            print(f"  Gene       : {gene}")
            print(f"  Product    : {product}")
            print(f"  Protein ID : {prot_id}")
            print(f"  Location   : {feat.location}")
            
            # Extract the nucleotide sequence of this CDS
            cds_seq = feat.extract(rec.seq)
            print(f"  CDS seq    : {cds_seq}")

---
## Part 6: BLAST — Sequence Similarity Search

### 🧬 Biological Context
BLAST (Basic Local Alignment Search Tool) is the most widely used bioinformatics tool. Given an unknown sequence, BLAST finds similar sequences in databases — letting you identify genes, classify organisms, and find evolutionary relatives.

**AMR use case**: BLAST an assembled contig against NCBI's AMR database to identify resistance genes.

In [ ]:
# ─────────────────────────────────────────────
# 6.1  Running BLAST Online (NCBIWWW)
# ─────────────────────────────────────────────
# This runs BLAST on NCBI's servers — requires internet

query_seq = "ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTTTTGCGGCATTTTGCCTTCCTGTTTTTGCTCACCCAGAAACGCTGGT"

print("Running BLAST... (this may take 30-120 seconds)")
print("Query: TEM-1 beta-lactamase fragment")
try:
    result_handle = NCBIWWW.qblast(
        program="blastn",      # blastn=DNA vs DNA, blastp=protein vs protein
        database="nt",         # nt=nucleotide, nr=non-redundant protein
        sequence=query_seq,
        hitlist_size=5         # return top 5 hits
    )
    blast_records = list(NCBIXML.parse(result_handle))
    result_handle.close()
    blast_ran = True
    print("BLAST complete.")
except Exception as e:
    blast_ran = False
    print(f"[Offline/Error] BLAST requires internet. Error: {e}")

In [ ]:
# ─────────────────────────────────────────────
# 6.2  Parsing BLAST Results
# ─────────────────────────────────────────────

if blast_ran and blast_records:
    for blast_record in blast_records:
        print(f"Query: {blast_record.query[:50]}")
        print(f"Database: {blast_record.database}")
        print(f"\n{'Rank':<5} {'Hit':<40} {'E-value':<12} {'Identity':<10} {'Score':<8}")
        print("-" * 80)
        
        for i, alignment in enumerate(blast_record.alignments[:5], 1):
            hsp = alignment.hsps[0]  # Best HSP (High-Scoring Pair)
            identity_pct = hsp.identities / hsp.align_length * 100
            print(f"{i:<5} {alignment.title[:40]:<40} {hsp.expect:<12.2e} "
                  f"{identity_pct:<10.1f} {hsp.score:<8.0f}")
else:
    # Demo output for offline mode
    print("=== Example BLAST Output (offline demo) ===")
    print(f"{'Rank':<5} {'Hit':<50} {'E-value':<12} {'Identity':<10}")
    print("-" * 80)
    demo_hits = [
        (1, "TEM-1 beta-lactamase [E. coli]",         "0.0",     "100.0"),
        (2, "TEM-2 beta-lactamase [E. coli]",         "1e-120",  "98.5"),
        (3, "TEM-30 beta-lactamase [K. pneumoniae]",  "1e-115",  "97.1"),
        (4, "SHV-1 beta-lactamase [K. pneumoniae]",   "2e-80",   "82.3"),
        (5, "OXA-1 beta-lactamase [P. aeruginosa]",   "5e-40",   "64.2"),
    ]
    for rank, hit, evalue, identity in demo_hits:
        print(f"{rank:<5} {hit:<50} {evalue:<12} {identity:<10}")
    print("\nInterpretation:")
    print("  E-value < 1e-5   → statistically significant hit")
    print("  Identity > 90%   → likely same or closely related gene")
    print("  Identity 70-90%  → related gene family")
    print("  Identity < 70%   → distantly related, interpret with caution")

In [ ]:
# ─────────────────────────────────────────────
# 6.3  Saving & Reusing BLAST Results
# ─────────────────────────────────────────────
# BEST PRACTICE: Always save your BLAST XML output.
# NCBI may not return identical results if you re-run later.

blast_xml_path = "/tmp/blast_results.xml"

# In real usage:
# result_handle = NCBIWWW.qblast("blastn", "nt", query_seq)
# with open(blast_xml_path, "w") as f:
#     f.write(result_handle.read())
# result_handle.close()
# 
# # Later, re-read from disk:
# with open(blast_xml_path) as f:
#     blast_records = list(NCBIXML.parse(f))

print("Best practice: save BLAST results to disk immediately after fetching.")
print("This ensures reproducibility — a core principle of good bioinformatics.")

---
## Part 7: Multiple Sequence Alignment (MSA)

### 🧬 Biological Context
Multiple Sequence Alignment (MSA) aligns ≥3 sequences simultaneously, allowing us to:
- Identify conserved regions (functionally important)
- Spot mutations across pathogen strains
- Build phylogenetic trees
- Visualize resistance mutations across AMR variants

In [ ]:
# ─────────────────────────────────────────────
# 7.1  Reading & Writing Alignments
# ─────────────────────────────────────────────

# Simulate a pre-computed MSA of beta-lactamase variants (CLUSTAL format)
# In real work this would be output from MUSCLE, MAFFT, or Clustal Omega

msa_content = """\
CLUSTAL W MSA output (beta-lactamase variants)

TEM-1      ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCC-TTTTTTGCGGCATTTTGCC
TEM-2      ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCC-TTTTTTGCGGCATTTTGCC
TEM-3      ATGAGTATTCAGCATTTCCGTGTCGCCCTTATTCCC-TTTTTTGCGGCATTTTGCC
SHV-1      ATGCGTATTCAACATTTCCGTGTCGCCCTTATTCCCATTTTTCGCGGCATTTTGCC
OXA-1      ATGAAAAAGAGTATTCAGATGATTTCCGTGTCGCCC-TTATTCCCTTTTTTGCGGC
                .*  *** ***** ******* ********* *   *  *** * **** **
"""

msa_path = "/tmp/beta_lactamase_msa.aln"
with open(msa_path, "w") as f:
    f.write(msa_content)

# Parse alignment
alignment = AlignIO.read(msa_path, "clustal")
print(f"Number of sequences : {len(alignment)}")
print(f"Alignment length    : {alignment.get_alignment_length()} bp")
print()
for record in alignment:
    print(f"{record.id:<10} {record.seq[:40]}...")

In [ ]:
# ─────────────────────────────────────────────
# 7.2  Alignment Analysis — Conservation Scoring
# ─────────────────────────────────────────────

import collections

def conservation_score(alignment, position):
    """Calculate conservation at a given column (0-based).
    Returns fraction of sequences with the most common character.
    """
    column = alignment[:, position]
    counter = collections.Counter(c for c in column if c != '-')
    if not counter:
        return 0
    return counter.most_common(1)[0][1] / len(alignment)

print("Positional conservation (first 20 positions):")
print(f"{'Pos':>5} {'Consensus':>10} {'Conservation':>14}")
print("-" * 35)
for i in range(min(20, alignment.get_alignment_length())):
    col = alignment[:, i]
    score = conservation_score(alignment, i)
    counter = collections.Counter(c for c in col if c != '-')
    most_common = counter.most_common(1)[0][0] if counter else '-'
    bar = '█' * int(score * 10)
    print(f"{i+1:>5} {most_common:>10} {score:>8.1%}  {bar}")

In [ ]:
# ─────────────────────────────────────────────
# 7.3  Substitution Matrix (BLOSUM/PAM)
# ─────────────────────────────────────────────
# Substitution matrices define the score for aligning one amino acid to another.
# BLOSUM62 is the standard for protein alignment.
# Higher score = more commonly observed substitution in real proteins.

from Bio.Align import substitution_matrices

# Load BLOSUM62
blosum62 = substitution_matrices.load("BLOSUM62")

print("BLOSUM62 matrix (selected entries):")
print(f"  Ala↔Ala : {blosum62['A','A']:>3}  (same AA, positive)")
print(f"  Ala↔Val : {blosum62['A','V']:>3}  (conservative, small positive)")
print(f"  Ala↔Trp : {blosum62['A','W']:>3}  (radical change, negative)")
print(f"  Glu↔Asp : {blosum62['E','D']:>3}  (similar charge, positive)")
print(f"  Glu↔Lys : {blosum62['E','K']:>3}  (opposite charge, negative)")

print(f"\nAvailable matrices: {substitution_matrices.load()}")

---
## Part 8: Phylogenetics

### 🧬 Biological Context
Phylogenetic trees show evolutionary relationships. In public health, they are used for:
- **Outbreak investigation**: tracing a SARS-CoV-2 variant cluster back to a source
- **AMR evolution**: tracking how resistance spreads between species
- **HIV molecular epidemiology**: linking transmission networks

In [ ]:
# ─────────────────────────────────────────────
# 8.1  Reading & Drawing Phylogenetic Trees
# ─────────────────────────────────────────────

from Bio import Phylo
from io import StringIO

# Newick format — the standard for storing trees
# This tree shows evolutionary relationships of beta-lactamase families
tree_newick = """\
((((TEM-1:0.01, TEM-2:0.01):0.05, TEM-3:0.04):0.1, 
   (SHV-1:0.08, SHV-12:0.09):0.12):0.2,
  ((KPC-2:0.15, KPC-3:0.14):0.2,
   (OXA-48:0.25, OXA-232:0.23):0.3):0.4,
  NDM-1:0.5);
"""

tree = Phylo.read(StringIO(tree_newick), "newick")

print("=== Phylogenetic Tree (Beta-Lactamase Families) ===")
print()
Phylo.draw_ascii(tree)

print(f"\nNumber of terminals (leaves) : {len(tree.get_terminals())}")
print(f"Number of internal nodes     : {len(tree.get_nonterminals())}")
print(f"Tree is rooted?              : {tree.rooted}")

In [ ]:
# ─────────────────────────────────────────────
# 8.2  Tree Operations
# ─────────────────────────────────────────────

# Get all leaf (terminal) names
leaves = [clade.name for clade in tree.get_terminals()]
print(f"Leaves: {leaves}")

# Find the path between two taxa
path = tree.get_path("TEM-1")
print(f"\nPath from root to TEM-1: {[str(c.name or 'internal') for c in path]}")

# Distance between taxa
d = tree.distance("TEM-1", "TEM-2")
print(f"\nEvolutionary distance TEM-1 ↔ TEM-2 : {d:.3f}")
d2 = tree.distance("TEM-1", "NDM-1")
print(f"Evolutionary distance TEM-1 ↔ NDM-1  : {d2:.3f}")
print("\n→ NDM-1 is much more distantly related (different enzyme class)")

# Check monophyly — are TEM variants a monophyletic group?
tem_variants = ["TEM-1", "TEM-2", "TEM-3"]
is_monophyletic = tree.is_monophyletic(tem_variants)
print(f"\nTEM variants monophyletic? {is_monophyletic}")

In [ ]:
# ─────────────────────────────────────────────
# 8.3  Building a Tree from Distance Matrix
# ─────────────────────────────────────────────

from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

# First we need an MSA
protein_msa_str = """\
>TEM-1
MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLS
>TEM-2
MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLS
>SHV-1
MRYIRLCIISLLATLPLAVHASPQPLEQIKLSESQLSGRVGMIEMDLASGRTLTAWRADERFPMMSTFKVLLCGAVLS-
>CTX-M-15
MMKFLRKLLPLLLVPVSFSTFSLGNAAHIQRKEAETLYSQLSRDMAQDGLRYVNELAKKYPDLPKTRIIENPQHKYKTK
"""

protein_msa_path = "/tmp/bl_protein_msa.fasta"
with open(protein_msa_path, "w") as f:
    f.write(protein_msa_str)

try:
    protein_alignment = AlignIO.read(protein_msa_path, "fasta")
    
    # Calculate distance matrix
    calculator = DistanceCalculator("identity")
    dm = calculator.get_distance(protein_alignment)
    print("Distance matrix:")
    print(dm)
    
    # Build NJ (Neighbor-Joining) tree
    constructor = DistanceTreeConstructor(calculator, "nj")
    nj_tree = constructor.build_tree(protein_alignment)
    print("\nNeighbor-Joining tree:")
    Phylo.draw_ascii(nj_tree)
    
except Exception as e:
    print(f"Tree building error (sequences may need padding): {e}")

---
## Part 9: Structural Bioinformatics (PDB)

### 🧬 Biological Context
Protein 3D structure determines function. For drug design, understanding the active site of a beta-lactamase (e.g., KPC-2) allows development of inhibitors. Biopython's `PDB` module parses structures from the Protein Data Bank.

In [ ]:
# ─────────────────────────────────────────────
# 9.1  Downloading PDB Structures
# ─────────────────────────────────────────────

from Bio.PDB import PDBParser, PDBIO, MMCIFParser
from Bio.PDB.PDBList import PDBList
import os

pdb_dir = "/tmp/pdb_files"
os.makedirs(pdb_dir, exist_ok=True)

try:
    pdbl = PDBList()
    # 1BT5 = TEM-1 beta-lactamase (the archetype AMR enzyme)
    pdb_file = pdbl.retrieve_pdb_file("1BT5", pdir=pdb_dir, file_type="pdb")
    print(f"Downloaded: {pdb_file}")
    pdb_downloaded = True
except Exception as e:
    print(f"[Offline mode] PDB download requires internet. Error: {e}")
    pdb_downloaded = False

In [ ]:
# ─────────────────────────────────────────────
# 9.2  Parsing & Exploring PDB Structures
# ─────────────────────────────────────────────
# The SMCRA hierarchy: Structure → Model → Chain → Residue → Atom

if pdb_downloaded:
    parser = PDBParser(QUIET=True)
    # Find the downloaded file
    pdb_files = [f for f in os.listdir(pdb_dir) if f.endswith(".ent")]
    if pdb_files:
        structure = parser.get_structure("TEM1", os.path.join(pdb_dir, pdb_files[0]))
        
        model = structure[0]  # First (and usually only) model
        print(f"Structure ID : {structure.id}")
        print(f"Models       : {len(list(structure.get_models()))}")
        print(f"Chains       : {[c.id for c in model.get_chains()]}")
        
        for chain in model:
            residues = list(chain.get_residues())
            atoms    = list(chain.get_atoms())
            print(f"  Chain {chain.id}: {len(residues)} residues, {len(atoms)} atoms")
else:
    print("=== Offline Demo: PDB Structure Analysis ===")
    print("""
Structure hierarchy (SMCRA):
  Structure (1BT5)
  └── Model [0]
      └── Chain [A]
          ├── Residue MET 1   (amino acid)
          │   ├── Atom N  (backbone)
          │   ├── Atom CA (alpha carbon)
          │   ├── Atom C  (backbone)
          │   ├── Atom O  (backbone)
          │   └── Atom CB (side chain)
          ├── Residue SER 70  (active site — catalytic serine!)
          └── ...

TEM-1 beta-lactamase (1BT5):
  263 residues in chain A
  Active site: Ser70 (nucleophile), Lys73, Ser130, Glu166
    """)

In [ ]:
# ─────────────────────────────────────────────
# 9.3  Active Site Residue Extraction
# ─────────────────────────────────────────────

if pdb_downloaded and pdb_files:
    structure = parser.get_structure("TEM1", os.path.join(pdb_dir, pdb_files[0]))
    model = structure[0]
    chain_a = model["A"]
    
    # Active site residues of TEM-1 beta-lactamase
    active_site_resnums = [70, 73, 130, 166, 234, 235, 236]
    
    print("Active site residues of TEM-1 beta-lactamase:")
    print(f"{'ResNum':<10} {'ResName':<10} {'Atoms':<8} {'Role'}")
    print("-" * 55)
    
    roles = {70: "Catalytic Ser (nucleophile)",
             73: "Lys (positions Ser70)",
             130: "Ser (H-bond network)",
             166: "Glu (activates water)",
             234: "Lys (oxyanion hole)",
             235: "Thr (oxyanion hole)",
             236: "Gly (oxyanion hole)"}
    
    for resnum in active_site_resnums:
        try:
            res = chain_a[(" ", resnum, " ")]
            n_atoms = len(list(res.get_atoms()))
            print(f"{resnum:<10} {res.resname:<10} {n_atoms:<8} {roles.get(resnum, '')}")
        except KeyError:
            print(f"{resnum:<10} {'N/A':<10} {'N/A':<8} {roles.get(resnum, '')}")

else:
    print("Active site analysis requires downloaded PDB structure.")
    print("Key active site residues for beta-lactamase drug design:")
    print("  Ser70  → Catalytic nucleophile (attacks beta-lactam ring)")
    print("  Lys73  → Positions Ser70")
    print("  Glu166 → Activates water for deacylation")
    print("  These residues are conserved → explain why they are drug targets")

In [ ]:
# ─────────────────────────────────────────────
# 9.4  Calculating Structural Properties
# ─────────────────────────────────────────────

if pdb_downloaded and pdb_files:
    import numpy as np
    from Bio.PDB import PPBuilder
    
    # Extract polypeptide chains
    ppb = PPBuilder()
    for pp in ppb.build_peptides(structure):
        sequence = pp.get_sequence()
        print(f"Polypeptide length : {len(sequence)} aa")
        print(f"Sequence (first 30): {str(sequence[:30])}...")
        
        # Phi/Psi angles (Ramachandran plot data)
        phi_psi = pp.get_phi_psi_list()
        print(f"\nPhi/Psi dihedral angles (first 5 residues):")
        for i, (phi, psi) in enumerate(phi_psi[:5]):
            phi_str = f"{phi:>8.2f}°" if phi else "       N/A"
            psi_str = f"{psi:>8.2f}°" if psi else "       N/A"
            print(f"  Residue {i+1}: phi={phi_str}  psi={psi_str}")
        break

else:
    print("Structural properties require PDB download.")
    print()
    print("Key structural analyses possible with Biopython:")
    print("  1. Phi/Psi angles  → Ramachandran plot analysis")
    print("  2. B-factors       → Flexibility/disorder")
    print("  3. RMSD            → Comparing two structures")
    print("  4. Contacts        → Residue-residue interaction map")
    print("  5. Accessible surface area (ASA) — buried vs exposed residues")

---
## Part 10: Population Genetics with Biopython

### 🧬 Biological Context
Population genetics tools help us understand how variants spread in populations — directly applicable to tracking AMR spread and pathogen evolution.

In [ ]:
# ─────────────────────────────────────────────
# 10.1  Population Genetics Basics
# ─────────────────────────────────────────────

from Bio.PopGen import GenePop

# Hardy-Weinberg Equilibrium — fundamental theorem of population genetics
# If p = allele freq of 'A', q = allele freq of 'a':
#   AA freq = p², Aa freq = 2pq, aa freq = q²
#   and p + q = 1

def hardy_weinberg(p):
    """Calculate expected HWE genotype frequencies."""
    q = 1 - p
    return {"AA": p**2, "Aa": 2*p*q, "aa": q**2}

print("Hardy-Weinberg Equilibrium Analysis")
print("Scenario: AMR allele frequency in a bacterial population")
print()
print(f"{'AMR allele freq (p)':<22} {'Resistant AA':>14} {'Intermediate Aa':>16} {'Sensitive aa':>14}")
print("-" * 70)
for p in [0.01, 0.05, 0.1, 0.2, 0.5, 0.9]:
    freqs = hardy_weinberg(p)
    print(f"{p:<22.2f} {freqs['AA']:>14.4f} {freqs['Aa']:>16.4f} {freqs['aa']:>14.4f}")

In [ ]:
# ─────────────────────────────────────────────
# 10.2  Nucleotide Diversity (Pi)
# ─────────────────────────────────────────────
# Pi measures average pairwise differences in a population
# High Pi in an AMR gene region = diverse resistance mechanisms

def nucleotide_diversity(sequences):
    """Calculate π (pi) — nucleotide diversity."""
    n = len(sequences)
    L = len(sequences[0])
    total_diffs = 0
    comparisons = 0
    
    for i in range(n):
        for j in range(i+1, n):
            diffs = sum(a != b for a, b in zip(sequences[i], sequences[j]))
            total_diffs += diffs
            comparisons += 1
    
    return total_diffs / (comparisons * L) if comparisons > 0 else 0

# Simulated AMR gene sequences from different isolates
isolates = [
    "ATGAGTATTCAACATTTCCGTGTCGCCCTT",  # TEM-1 reference
    "ATGAGTATTCAACATTTCCGTGTCGCCCTT",  # identical
    "ATGAGTATTCAGCATTTCCGTGTCGCCCTT",  # 1 SNP (position 13)
    "ATGAGTATCCAACATTTCCGTGTCGCCCTT",  # 1 SNP (position 9)
    "ATGAGTATTCAACATTTCCGTGTCGCCATT",  # 1 SNP (position 29)
]

pi = nucleotide_diversity(isolates)
print(f"Number of sequences   : {len(isolates)}")
print(f"Sequence length       : {len(isolates[0])} bp")
print(f"Nucleotide diversity π: {pi:.4f}")
print(f"  → {pi*100:.2f}% average pairwise difference")
print("  → Low π suggests recent clonal expansion (common in outbreak scenarios)")

---
## Part 11: Real-World Mini-Pipeline

### 🧬 Biological Context: AMR Gene Detection Pipeline

**Scenario**: You receive 5 assembled contigs from a clinical *K. pneumoniae* isolate suspected of being carbapenem-resistant. Your task:
1. Parse the sequences
2. Calculate basic statistics
3. Search for ORFs
4. Calculate GC content (AMR genes often have atypical GC)
5. Generate a summary report

This simulates a real first-pass analysis in a public health genomics lab.

In [ ]:
# ─────────────────────────────────────────────
# 11.1  Setup — Simulated Contigs
# ─────────────────────────────────────────────

import json
from datetime import datetime

# Simulated assembled contigs from a carbapenem-resistant K. pneumoniae isolate
contig_data = [
    SeqRecord(Seq(
        "ATGAGCATTCAGTTTCGTGTTGCACTGATCCCATTTTTCGCAGCGTTTTGCTTATCCCCAGTCTTCGCTG"
        "CACCCAGAAACGCTGGTAAAAGTATTTAATCTTTTTAATCAACAGGATTTGAGCTGGCATCAGTCGTTAA"
        "TCAGACGGCGTTTAAGGCGATTAAGCAGCGTGAGCAGTTGCTGGCGGAAGCCGCCGCGTTGCAGCAGCAG"
    ), id="CONTIG_001", description="KPC-2 containing contig"),
    SeqRecord(Seq(
        "ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTTTTGCGGCATTTTGCCTTCCTGTTTTTGCTC"
        "ACCCAGAAACGCTGGTAAAAGTATTTAATCTTTTTAATCAACAGGATTTGAGCTGGCATCAGTCGTTAAT"
    ), id="CONTIG_002", description="TEM-1 beta-lactamase contig"),
    SeqRecord(Seq(
        "GCATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG"
        "ATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGAT"
    ), id="CONTIG_003", description="Unknown contig"),
    SeqRecord(Seq(
        "ATGCGTAAAGGAGAAGAACTTTTCACTGGAGTTGTCCCAATTCTTGTTGAATTAGATGGTGATGTTAATGG"
        "GCACAAATTTTCTGTCAGTGGAGAGGGTGAAGGTGATGCAACATACGGAAAACTTACCCTTAAATTTATT"
    ), id="CONTIG_004", description="GFP-like gene contig"),
    SeqRecord(Seq(
        "ATGAAAATTCTTGAAATCAGCCAAGATCGTCAGCGTCAGCAGCAGCAGCAGCAGCAGCAGCAACAGCAGC"
        "AGCAGCAGCAGCAAACAGCAGCTGCAGCAGCAACAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGC"
    ), id="CONTIG_005", description="Repeat-containing contig"),
]

print(f"Loaded {len(contig_data)} contigs")
print(f"Total bases: {sum(len(c.seq) for c in contig_data):,}")

In [ ]:
# ─────────────────────────────────────────────
# 11.2  Sequence Statistics
# ─────────────────────────────────────────────

from Bio.SeqUtils import gc_fraction

print("=" * 70)
print(" CONTIG QUALITY SUMMARY")
print("=" * 70)
print(f"{'Contig':<15} {'Length':>8} {'GC%':>7} {'ORFs':>6} {'Flag':<20}")
print("-" * 70)

results = []
for rec in contig_data:
    gc = gc_fraction(rec.seq)
    n_orfs = len(find_orfs(rec.seq, min_length=60))
    
    # Flag unusual GC content (typical bacterial genome ~40-65%)
    if gc < 0.30 or gc > 0.75:
        flag = "⚠ Atypical GC"
    elif n_orfs == 0:
        flag = "⚠ No ORFs found"
    else:
        flag = "✓ OK"
    
    results.append({
        "id": rec.id, "length": len(rec.seq),
        "gc": gc, "orfs": n_orfs, "flag": flag,
        "description": rec.description
    })
    print(f"{rec.id:<15} {len(rec.seq):>8,} {gc:>7.1%} {n_orfs:>6} {flag:<20}")

print("-" * 70)
total_len = sum(r['length'] for r in results)
avg_gc = sum(r['gc'] for r in results) / len(results)
print(f"{'TOTAL':<15} {total_len:>8,} {avg_gc:>7.1%}")

In [ ]:
# ─────────────────────────────────────────────
# 11.3  AMR Gene Signature Screen
# ─────────────────────────────────────────────
# Search for known AMR gene signature sequences
# In real pipelines: use CARD RGI, AMRFinderPlus, or ResFinder

AMR_SIGNATURES = {
    "KPC-2_signature"  : "ATGAGCATTCAGTTTCGT",   # KPC carbapenemase start
    "TEM-1_signature"  : "ATGAGTATTCAACATTTCCGT", # TEM beta-lactamase start
    "CTX-M_signature"  : "ATGVGGTTACAATGCTTG",    # CTX-M ESBL
    "NDM-1_signature"  : "ATGGAATTGCCCAATATTAT",   # NDM carbapenemase
    "OXA-48_signature" : "ATGAAAGTATTAAAAATGTTAG", # OXA-48 carbapenemase
}

print("=" * 70)
print(" AMR GENE SIGNATURE SCREEN")
print("=" * 70)
print(f"{'Contig':<15} {'AMR Gene':<25} {'Match':>6} {'Position':>10}")
print("-" * 60)

detections = []
for rec in contig_data:
    for amr_name, signature in AMR_SIGNATURES.items():
        pos = str(rec.seq).upper().find(signature.upper())
        if pos >= 0:
            detections.append({"contig": rec.id, "gene": amr_name, "pos": pos})
            print(f"{rec.id:<15} {amr_name:<25} {'HIT':>6} {pos:>10}")
        # Check reverse complement
        rc_pos = str(rec.seq.reverse_complement()).upper().find(signature.upper())
        if rc_pos >= 0:
            detections.append({"contig": rec.id, "gene": amr_name, "pos": f"RC:{rc_pos}"})
            print(f"{rec.id:<15} {amr_name:<25} {'RC HIT':>6} {rc_pos:>10}")

if not detections:
    print("No exact signature matches (try partial matching or BLAST for real data)")

print("-" * 60)
print(f"Total AMR detections: {len(detections)}")

In [ ]:
# ─────────────────────────────────────────────
# 11.4  Generate JSON Report
# ─────────────────────────────────────────────
# Reproducible output — always save your results!

report = {
    "analysis_date"  : datetime.now().isoformat(),
    "tool"           : "Biopython AMR Screen v1.0",
    "sample_id"      : "KPN_2024_001",
    "organism"       : "Klebsiella pneumoniae (suspected)",
    "total_contigs"  : len(contig_data),
    "total_bases"    : sum(len(c.seq) for c in contig_data),
    "mean_gc"        : round(avg_gc, 4),
    "amr_detections": detections,
    "contigs": [
        {"id": r["id"], "length": r["length"],
         "gc": round(r["gc"], 4), "orfs": r["orfs"],
         "flag": r["flag"]}
        for r in results
    ]
}

report_path = "/tmp/amr_screen_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

---
## Quick Reference Cheatsheet

| Task | Code |
|---|---|
| Create a sequence | `seq = Seq("ATGCGT")` |
| Reverse complement | `seq.reverse_complement()` |
| Transcribe | `seq.transcribe()` |
| Translate | `seq.translate(table=11, to_stop=True)` |
| GC content | `gc_fraction(seq)` |
| Read FASTA | `SeqIO.parse("file.fa", "fasta")` |
| Read FASTQ | `SeqIO.parse("file.fq", "fastq")` |
| Read GenBank | `SeqIO.parse("file.gb", "genbank")` |
| Write sequences | `SeqIO.write(records, "out.fa", "fasta")` |
| Convert formats | `SeqIO.convert("in.gb", "genbank", "out.fa", "fasta")` |
| Dict index | `SeqIO.to_dict(SeqIO.parse("file.fa", "fasta"))` |
| NCBI search | `Entrez.esearch(db="nucleotide", term="...")` |
| NCBI fetch | `Entrez.efetch(db="nucleotide", id="NC_000913", rettype="gb")` |
| BLAST online | `NCBIWWW.qblast("blastn", "nt", sequence)` |
| Parse BLAST | `NCBIXML.parse(result_handle)` |
| Read alignment | `AlignIO.read("file.aln", "clustal")` |
| Read tree | `Phylo.read(handle, "newick")` |
| Draw tree | `Phylo.draw_ascii(tree)` |
| Protein analysis | `ProteinAnalysis(str(seq))` |
| MW calculation | `molecular_weight(seq, "DNA")` |
| Melting temp | `MeltingTemp.Tm_NN(seq)` |

---

## Practice Exercises 🎯

**Beginner**
1. Write a function that takes a DNA sequence and returns its reverse complement, GC content, and predicted melting temperature.
2. Parse the E. coli K-12 genome FASTA file from NCBI and calculate total genome GC content.

**Intermediate**
3. Write a script that reads a multi-FASTA file of AMR genes and outputs a table with GC%, length, and translated protein.
4. Use Entrez to search for all KPC beta-lactamase sequences and compute nucleotide diversity.

**Advanced**
5. Build a mini-pipeline that: (1) fetches 10 TEM beta-lactamase sequences from NCBI, (2) aligns them, (3) builds a phylogenetic tree, and (4) identifies conserved positions.
6. Write an AMR gene detector that takes assembled contigs and reports: ORFs > 100 aa, their GC content, and BLAST hits against an AMR database.

---
*Notebook developed for the Computational Biology Learning Program | Biopython v1.87+*